# SQL Dataset Merge + Cleaning

Bu notebook, iki farkli SQL dataset'ini ortak formata getirir, birlestirir,
EDA ile uyumlu kolon temizligi ve etiket normalizasyonu uygular: `true` / `yes` / `positive` /
`malicious` gibi degerler 1'e, `false` / `no` / `negative` / `benign` gibi degerler 0'a map'lenir;
bilinmeyen etiketler atilir; `normalize_sql` ile `normalized_query` uretilir ve `(normalized_query, Label)` yinelenenleri silinir. Sonunda yeni bir CSV dosyasi olusturur.

In [1]:
import math
import os
import re

import pandas as pd

## 1) Veri Kaynaklarini Yukle, Karistir ve Birlestir

- Her iki CSV `encoding="UTF-8"` ile okunur (EDA ile ayni).
- `looking for beter dataset/dataset.csv` dosyasindan sadece `full_query` ve `label` alinir.
- Kolon adlari `Sentence` ve `Label` olarak degistirilir.
- Ilk dataset tamamen rastgele karistirilir ve indeks sifirlanir (`random_state=42`).
- `SQLiV3.csv`: EDA'daki gibi `Unnamed: 2` ve `Unnamed: 3` kolonlari kaldirilir, ardindan `Sentence` ve `Label` secilir.
- Iki dataset birlestirilir.
- Birlestirilen yeni dataset son kez tamamen rastgele karistirilir ve indeks yeniden sifirlanir (`random_state=42`).

In [2]:
PATH_BETTER = "./data/looking for beter dataset/dataset.csv"  # Ilk dataset dosya yolu
PATH_SQLIV3 = "./data/SQLiV3.csv"  # Ikinci dataset dosya yolu
OUTPUT_PATH = "./data/merged_cleaned_preprocessed.csv"  # Cikti dosya yolu

# 1) Ilk dataseti oku (EDA ile ayni encoding)
better_df = pd.read_csv(PATH_BETTER, low_memory=False, encoding="UTF-8")
better_df = better_df[["full_query", "label"]].copy()
better_df = better_df.rename(columns={"full_query": "Sentence", "label": "Label"})
better_df = better_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 2) SQLiV3: Dataset_EDA.ipynb ile ayni — once Unnamed kolonlari, sonra Sentence/Label
sqliv3_df = pd.read_csv(PATH_SQLIV3, encoding="UTF-8")
sqliv3_df = sqliv3_df.drop(columns=["Unnamed: 2", "Unnamed: 3"], errors="ignore")

if "Sentence" not in sqliv3_df.columns or "Label" not in sqliv3_df.columns:
    raise ValueError("SQLiV3.csv dosyasinda 'Sentence' ve 'Label' kolonlari bulunamadi.")

sqliv3_df = sqliv3_df[["Sentence", "Label"]].copy()

# 3) Iki dataseti birlestir ve karistir
merged_df = pd.concat([sqliv3_df, better_df], ignore_index=True)
merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("SQLiV3 shape:", sqliv3_df.shape)
print("Better dataset (first shuffle) shape:", better_df.shape)
print("Merged dataset (final shuffle) shape:", merged_df.shape)
merged_df.head()

SQLiV3 shape: (30919, 2)
Better dataset (first shuffle) shape: (3688977, 2)
Merged dataset (final shuffle) shape: (3719896, 2)


,Sentence,Label
0,"UPDATE regions SET keywords = ""Airports in Kou...",0
1,"UPDATE airport SET home_link = ""http://www.kil...",0
2,"DELETE FROM airport_frequencies WHERE type = ""...",1
3,"INSERT INTO airport (ident, type, name, latitu...",0
4,"SELECT r.name AS region_name, COUNT(a.id) AS a...",0


## 2) Data Cleaning

- `Label` icin baslangicta NaN satirlar kaldirilir.
- Etiketler 0/1 tamsayiya **map** edilir: `0`/`1`, `true`/`yes`, `false`/`no`, `positive`/`negative`,
  `pos`/`neg`, `+1`/`-1`, `sqli`/`malicious`/`attack`, `benign`/`normal`/`safe` ve benzeri sinif adlari
  (kucuk harf duyarsiz); sayisal `1.0`/`0.0` stringleri de kabul edilir. Map edilemeyen etiketler silinir.
- `Sentence` metni degistirilmez; `normalize_sql` ile `normalized_query` (kucuk harf, `<IP>`, `<HEX>`, `<NUM>` maskeleri) uretilir.
- Ayni `(normalized_query, Label)` tekrarlari `drop_duplicates` ile atilir (sayisal/IP/hex varyasyonlari birlestirilir).

In [3]:
def map_label_to_binary(raw):
    """true/yes/positive/sqli -> 1; false/no/negative/benign -> 0; taninmayan -> NaN."""
    if pd.isna(raw):
        return float("nan")
    if isinstance(raw, bool):
        return int(raw)
    if isinstance(raw, int):
        if raw == 1:
            return 1
        if raw == 0:
            return 0
        return float("nan")
    if isinstance(raw, float):
        if math.isnan(raw):
            return float("nan")
        if raw == 1.0:
            return 1
        if raw == 0.0:
            return 0
        return float("nan")
    s = str(raw).strip().lower()
    if s in {
        "1",
        "1.0",
        "+1",
        "true",
        "yes",
        "t",
        "y",
        "positive",
        "pos",
        "sqli",
        "sql_injection",
        "malicious",
        "attack",
        "harmful",
        "unsafe",
        "bad",
        "suspicious",
        "anomaly",
        "anomalous",
        "class_1",
        "label_1",
        "class1",
        "label1",
        "malware",
        "exploit",
    }:
        return 1
    if s in {
        "0",
        "0.0",
        "-1",
        "false",
        "no",
        "f",
        "n",
        "negative",
        "neg",
        "benign",
        "normal",
        "safe",
        "clean",
        "legitimate",
        "harmless",
        "good",
        "non-malicious",
        "non_malicious",
        "nonmalicious",
        "class_0",
        "label_0",
        "class0",
        "label0",
        "innocuous",
        "valid",
        "acceptable",
    }:
        return 0
    try:
        v = float(s)
        if v == 1.0:
            return 1
        if v == 0.0:
            return 0
    except ValueError:
        pass
    return float("nan")


def normalize_sql(query):
    """Metni kucuk harfe cevir; IP, hex ve ardisik sayilari maskele."""
    text = str(query).lower()
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", "<IP>", text)
    text = re.sub(r"0x[0-9a-f]+", "<HEX>", text)
    text = re.sub(r"\d+", "<NUM>", text)
    return text


clean_df = merged_df.copy()
clean_df = clean_df.dropna(subset=["Label"]).copy()
clean_df["Label"] = clean_df["Label"].apply(map_label_to_binary)
clean_df = clean_df.dropna(subset=["Label"]).copy()
clean_df["Label"] = clean_df["Label"].astype(int)
clean_df["normalized_query"] = clean_df["Sentence"].apply(normalize_sql)
clean_df = clean_df.drop_duplicates(subset=["normalized_query", "Label"]).reset_index(drop=True)

print("Cleaned shape:", clean_df.shape)
print(clean_df["Label"].value_counts(dropna=False))
clean_df.head()

Cleaned shape: (1665053, 3)
Label
0    1490101
1     174952
Name: count, dtype: int64


,Sentence,Label,normalized_query
0,"UPDATE regions SET keywords = ""Airports in Kou...",0,"update regions set keywords = ""airports in kou..."
1,"UPDATE airport SET home_link = ""http://www.kil...",0,"update airport set home_link = ""http://www.kil..."
2,"DELETE FROM airport_frequencies WHERE type = ""...",1,"delete from airport_frequencies where type = ""..."
3,"INSERT INTO airport (ident, type, name, latitu...",0,"insert into airport (ident, type, name, latitu..."
4,"SELECT r.name AS region_name, COUNT(a.id) AS a...",0,"select r.name as region_name, count(a.id) as a..."


## 3) Son dataseti karistir ve CSV kaydet

Tum temizlikten sonra satirlar tekrar rastgele siralanir (`sample`, `random_state=42`) — etiketler
homojen dagilsin ve train/val split oncesi bloklanma azalsin diye. Ardından UTF-8 CSV yazilir.

In [4]:
# Cikti klasorunu garanti et
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

final_df = clean_df[["Sentence", "normalized_query", "Label"]].copy()
# Son adim: tum islemlerden sonra homojen siralama (onceki shuffle ile ayni tohum)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Yeni CSV olusturuldu:")
print(OUTPUT_PATH)
print("Satir sayisi:", len(final_df))
final_df.head(5)

Yeni CSV olusturuldu:
./data/merged_cleaned_preprocessed.csv
Satir sayisi: 1665053


,Sentence,normalized_query,Label
0,"SELECT n.type, c.name as country_name, COUNT(n...","select n.type, c.name as country_name, count(n...",0
1,"SELECT a.id, a.ident, a.name, a.latitude_deg, ...","select a.id, a.ident, a.name, a.latitude_deg, ...",0
2,UPDATE airport_frequencies SET type = 'TWR' WH...,update airport_frequencies set type = 'twr' wh...,0
3,"SELECT name, 'airport' AS type FROM airport WH...","select name, 'airport' as type from airport wh...",0
4,"SELECT name, 'airport' AS entity_type FROM air...","select name, 'airport' as entity_type from air...",0


In [5]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1665053 entries, 0 to 1665052
Data columns (total 3 columns):
 #   Column            Non-Null Count    Dtype
---  ------            --------------    -----
 0   Sentence          1665053 non-null  str  
 1   normalized_query  1665053 non-null  str  
 2   Label             1665053 non-null  int64
dtypes: int64(1), str(2)
memory usage: 38.1 MB
